In [1]:

import logging
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
# from decimal import Decimal
from datetime import datetime, timezone, timedelta

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

from typing import Optional, Tuple



In [2]:
# ============================================================================
# [START] - Polars Display Configurations
# ============================================================================

pl.Config.set_tbl_rows(1000)      # show up to 100 rows
pl.Config.set_tbl_cols(50)       # show up to 50 columns
# pl.Config.set_tbl_width_chars(0) # 0 = no width limit
pl.Config.set_tbl_hide_dataframe_shape(False)  # keep shape info

# [ DEFAULT DECIMAL DISPLAY ]
# # Option 1: Set display precision globally
# pl.Config.set_fmt_float("full")

# Option 2: Set specific decimal places
# pl.Config.set_float_precision(7)

# # Option 3: Check individual values
# print(result["price_decimal"][7])  # Will show: 0.0070995

# ============================================================================
# [END] - Polars Display Configurations
# ============================================================================

polars.config.Config

In [3]:

class AvailableDatasets:
    """Available Datasets, returns request ID"""

    FULL_YEAR_2024__CME_JPY_V: str = "GLBX-20251028-HAE9P7SP3U"
    FULL_YEAR_2023__CME_JPY_V: str = "GLBX-20251116-MJ5JKQA9A4"
    FULL_YEAR_2022__CME_JPY_V: str = "GLBX-20251115-DJHUGKSLWB"


In [4]:

# WORKING DIRECTORY
MAIN_DIR = Path("/Users/number6/Projects/Bimini/")

# HISTORICAL DATA
HISTORICAL_DATA = MAIN_DIR / "0_data" / "historical"

# IMPORT PATH
PARQUET_DIR = HISTORICAL_DATA / "parquet"

# EXPORT PATH
EXPORT_DIR = MAIN_DIR / "journal_results"


In [5]:


"""

?

"""



TICK_SIZE = 0.00001




# Convert START_DATE and END_DATE to nanosecond epochs (integers)
START_DATE_NS = int(
    datetime(
        year=2022,
        month=11,
        day=7,
        hour=23,
        minute=0,
        second=0,
        tzinfo=timezone.utc,
    ).timestamp() * 1e9
)

END_DATE_NS = int(
    datetime(
        year=2022,
        month=11,
        day=12,
        hour=18,
        minute=0,
        second=0,
        tzinfo=timezone.utc,
    ).timestamp() * 1e9
)


hrf_start_dt = datetime.fromtimestamp(START_DATE_NS / 1e9, tz=timezone.utc)
hrf_end_dt = datetime.fromtimestamp(END_DATE_NS / 1e9, tz=timezone.utc)

print("START_DATE_NS", hrf_start_dt.strftime("%Y-%m-%d %H:%M:%S %Z"))
print("END_DATE_NS", hrf_end_dt.strftime("%Y-%m-%d %H:%M:%S %Z"))


# [ Rolling average container ]
# ( Container_Size > Search_Range )
# Thus only consider end time for each new interval of the rolling average that would be available in the time frame




START_DATE_NS 2022-11-07 23:00:00 UTC
END_DATE_NS 2022-11-12 18:00:00 UTC


In [6]:

BUCKET_DIR = Path("/Users/number6/Projects/Bimini/pool_vb100_fb30/2313_Nov_21_25/bucket/fixed/bucket_fixed_consolidated.parquet")

# Data files to glob for processing
BUCKET_DATA_GLOB = pl.scan_parquet(str(BUCKET_DIR))

# Schema? ...


In [7]:
VOLUME_BAR_DIR = Path("/Users/number6/Projects/Bimini/pool_vb100_fb30/2313_Nov_21_25/volumebar/raw/volumebar_raw_consolidated.parquet")

# Data files to glob for processing
VOLUME_BAR_DATA_GLOB = pl.scan_parquet(str(VOLUME_BAR_DIR))

# Schema? ...



### Winzorize VolumeBars

In [8]:
# ...

### Generate Custom Buckets from VolumeBars

In [9]:
BUCKET_SIZE__IN_VOLUME_BARS = 50


buckets_from_volume_bars = (
    
    VOLUME_BAR_DATA_GLOB
    .with_columns(
        bucket=pl.int_range(pl.len()) // BUCKET_SIZE__IN_VOLUME_BARS
    )
    .group_by("bucket").agg(
        passive_vwap=pl.col("passive_midprice").mean(),
        volume_total_sum=pl.col("volume_total").sum(),

        active_imbalance_signed=pl.col("active_imbalance_signed").sum(),
        passive_imbalance_signed=pl.col("passive_imbalance_signed").sum(),

        time_elapsed_ns_total=pl.col("time_elapsed_ns").sum(),
        start_ts_ns=pl.col("start_ts_ns").first(),
        end_ts_ns=pl.col("end_ts_ns").last(),

        contract_roll_any=pl.col("contract_roll").any(),
        gap_return_any=pl.col("gap_return").any(),

    )
    .with_columns(
        prev_passive_vwap=pl.col("passive_vwap").shift(1),
    )
    .filter([
        pl.col("prev_passive_vwap").is_not_null(),
    ])
    .with_columns(
        price_delta=(pl.col("passive_vwap") - pl.col("passive_vwap").shift(1)),
        pct_return=(pl.col("passive_vwap") - pl.col("passive_vwap").shift(1)) / pl.col("passive_vwap").shift(1),
        log_return=pl.col("passive_vwap").log() - pl.col("passive_vwap").shift(1).log(),
    )
    .sort("bucket")
    .drop("bucket")
    # .select([
    #     "delta_midpoint_vwap_log",
    #     "buy_vwap",
    #     "sell_vwap",
    #     "total_vwap",
    #     "spread",
    #     "prev_total_vwap",
    #     "total_vwap_change",
    #     "pct_return",
    #     "log_return"
    # ])

)

### Create Metric FROM Bucket

In [10]:

"""

Create metric

"""




# Bucket Lookback Count
ROLLING_WINDOW_SIZE: int = 65 #50 #250

metric_from_artificial_buckets = (
    buckets_from_volume_bars
    # (1) [ SUM values over the ROLLING_WINDOW_SIZE ]
    .with_columns([

        # Summary Data
        pl.col("time_elapsed_ns_total").rolling_sum(window_size=ROLLING_WINDOW_SIZE).alias("time_elapsed_ns_total_rolling"),
        pl.col("volume_total_sum").rolling_sum(window_size=ROLLING_WINDOW_SIZE).alias("sum__volume_total_sum"),
        pl.col("start_ts_ns"),
        pl.col("end_ts_ns"),

        # Returns...
        # pl.col("volume_total_sum").rolling_sum(window_size=ROLLING_WINDOW_SIZE).alias("sum__volume_total_sum"),

        # Sum of Completely Absolute Value (Seems Wrong)
        pl.col("active_imbalance_signed").rolling_sum(window_size=ROLLING_WINDOW_SIZE).alias("sum__active_imbalance_abs_wrong"),
        pl.col("passive_imbalance_signed").rolling_sum(window_size=ROLLING_WINDOW_SIZE).alias("sum__passive_imbalance_abs_wrong"),

        # Sum of Actual Absolute Value (Seems Right?)
        (pl.col("active_imbalance_signed").rolling_sum(window_size=ROLLING_WINDOW_SIZE)).abs().alias("sum__active_imbalance_abs"),
        (pl.col("passive_imbalance_signed").rolling_sum(window_size=ROLLING_WINDOW_SIZE)).abs().alias("sum__passive_imbalance_abs"),

        # Signed Values
        pl.col("active_imbalance_signed").rolling_sum(window_size=ROLLING_WINDOW_SIZE).alias("sum__active_imbalance_signed"),
        pl.col("passive_imbalance_signed").rolling_sum(window_size=ROLLING_WINDOW_SIZE).alias("sum__passive_imbalance_signed"),
        
    ])
    # (2) [ FILTER/REMOVE the first n rows that have incomplete windows ]
    .filter(

        # Sum of Completely Absolute Value (Seems Wrong)
        pl.col("sum__active_imbalance_abs_wrong").is_not_null(),
        pl.col("sum__passive_imbalance_abs_wrong").is_not_null(),

        # Sum of Actual Absolute Value (Seems Right?)
        pl.col("sum__active_imbalance_abs").is_not_null(),
        pl.col("sum__passive_imbalance_abs").is_not_null(),

        # Signed Value
        pl.col("sum__active_imbalance_signed").is_not_null(),
        pl.col("sum__passive_imbalance_signed").is_not_null(),

    )
    .with_columns([

        # [ Normalize Imbalances with Volumes ]

        # Sum of Completely Absolute Value (Seems Wrong)
        (pl.col("sum__active_imbalance_abs_wrong") / pl.col("sum__volume_total_sum")).alias("normalized__active_imbalance_abs_wrong"),
        (pl.col("sum__passive_imbalance_abs_wrong") / pl.col("sum__volume_total_sum")).alias("normalized__passive_imbalance_abs_wrong"),

        # Sum of Actual Absolute Value (Seems Right?)
        (pl.col("sum__active_imbalance_abs") / pl.col("sum__volume_total_sum")).alias("normalized__active_imbalance_abs"),
        (pl.col("sum__passive_imbalance_abs") / pl.col("sum__volume_total_sum")).alias("normalized__passive_imbalance_abs"),

        # Signed Value
        (pl.col("sum__active_imbalance_signed") / pl.col("sum__volume_total_sum")).alias("normalized__active_imbalance_signed"),
        (pl.col("sum__passive_imbalance_signed") / pl.col("sum__volume_total_sum")).alias("normalized__passive_imbalance_signed"),

    ])
    .with_columns([

        # ( For "percentile" value 0-100 )
        # (pl.col("total_active_imbalance_signed").rank(method="average").truediv(pl.len()) * 100).alias("ecdf__total_active_imbalance_signed"),

        # [ CDF Calculations]

        # Sum of Completely Absolute Value (Seems Wrong)
        (pl.col("normalized__active_imbalance_abs_wrong").rank(method="average").truediv(pl.len())).alias("ecdf__normalized__active_imbalance_abs_wrong"),
        (pl.col("normalized__passive_imbalance_abs_wrong").rank(method="average").truediv(pl.len())).alias("ecdf__normalized__passive_imbalance_abs_wrong"),

        # Sum of Actual Absolute Value (Seems Right?)
        (pl.col("normalized__active_imbalance_abs").rank(method="average").truediv(pl.len())).alias("ecdf__normalized__active_imbalance_abs"),
        (pl.col("normalized__passive_imbalance_abs").rank(method="average").truediv(pl.len())).alias("ecdf__normalized__passive_imbalance_abs"),

        # Signed Value
        (pl.col("normalized__active_imbalance_signed").rank(method="average").truediv(pl.len())).alias("ecdf__normalized__active_imbalance_signed"),
        (pl.col("normalized__passive_imbalance_signed").rank(method="average").truediv(pl.len())).alias("ecdf__normalized__passive_imbalance_signed"),


    ])
    .with_columns([
        
        (pl.col("ecdf__normalized__active_imbalance_abs").log()).alias("ecdf__normalized__active_imbalance_abs_log"),
        (pl.col("ecdf__normalized__passive_imbalance_abs").log()).alias("ecdf__normalized__passive_imbalance_abs_log"),

    ])
    .with_columns([
        
        pl.col("ecdf__normalized__active_imbalance_abs_log").alias("ln_vpin"),

    ])

)

### Find the Surface